In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2,3: For states
# 4,5: Ancillary qubits

In [ ]:
from qiskit import QuantumCircuit
from numpy import sqrt
from qiskit.circuit.library import StatePreparation

def circuit_init():
    qc = QuantumCircuit(6)
    return qc

def PREP(qc):
    desired_vector = [
        1/sqrt(3), 0, 0, 0, 
        1/(4*sqrt(3)), 1/4, 1/4, sqrt(3)/4, 
        1/(4*sqrt(3)), -1/4, -1/4, sqrt(3)/4, 
        0, 0, 0, 0
    ]
    prep = StatePreparation(desired_vector)
    qc.append(prep,[3,2,1,0])
    return qc

In [ ]:
from qiskit.circuit.library.standard_gates import RXGate,RYGate, RZGate
from qiskit.circuit.library import U3Gate
from qiskit.circuit.library import MCXGate
from qiskit.circuit.library.standard_gates import XGate
from pennylane.templates.state_preparations.mottonen import compute_theta, gray_code
from numpy import array, log2


def UU(qc, wires, params):
    qc.append(U3Gate(params[0],params[1],params[2]),[wires[0]])
    qc.append(U3Gate(params[3],params[4],params[5]),[wires[1]])


def RR_Z(qc, wires, params):
    qc.append(RZGate(params[0]),[wires[0]])
    qc.append(RZGate(params[1]),[wires[1]])

def CUU(qc,wires, params, state='1'):
    qc.append(U3Gate(params[0],params[1],params[2]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(U3Gate(params[3],params[4],params[5]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CRR_Z(qc,wires, params, state='1'):
    qc.append(RZGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]
    qc.append(RZGate(params[1]).control(1,ctrl_state = state[::-1]), [wires[0], wires[2]])  # Control: wires[0] Target: wires[2]
    return qc

def CR_Y(qc, wires, params, state = '1'):
    qc.append(RYGate(params[0]).control(1,ctrl_state = state[::-1]), [wires[0], wires[1]])  # Control: wires[0] Target: wires[1]


def CCUU(qc,wires, params, state='11'):
    qc.append(U3Gate(params[0],params[1],params[2]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(U3Gate(params[3],params[4],params[5]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]

def CCRR_Z(qc, wires, params, state = '11'):
    qc.append(RZGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    qc.append(RZGate(params[1]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[3]])  # Control: wires[0] wires[1]
    return qc
    

def CCR_Y(qc, wires, params, state = '11'):
    qc.append(RYGate(params[0]).control(2, ctrl_state = state[::-1]), [wires[0], wires[1],wires[2]])  # Control: wires[0] wires[1]
    return qc

def U_CCR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-1
    code = gray_code(2)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.ry(params[i],wires[2])
        qc.cx(wires[num_Ucontrols-1 - control_order[i]],wires[2])
    
    return qc

def U_CCcR_decom(qc,wires,params):
    # Make sure that control qubits should be increasing order
    params = array(params)
    params = compute_theta(params) #can be on/off

    num_Ucontrols = len(wires)-2
    code = gray_code(2)
    n_selections = len(code)
    control_order = [int(log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

    for i in range(n_selections):
        qc.cry(params[i],wires[2],wires[3])
        CCX(qc,[wires[num_Ucontrols-1 - control_order[i]],wires[2],wires[3]],'11')
    
    return qc

def CCX(qc, wires, state = '11'):
    mcx_gate = MCXGate(num_ctrl_qubits=2,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2]])
    return qc

def CCCX(qc, wires, state = '111'):
    mcx_gate = MCXGate(num_ctrl_qubits=3,ctrl_state=state[::-1])
    qc.append(mcx_gate, [wires[0], wires[1],wires[2], wires[3]])
    return qc

def CX01(qc, wires):
    qc.append(XGate().control(1, ctrl_state='1'),[wires[0], wires[1]])
    return qc


In [ ]:
from qiskit.circuit.library.standard_gates import RYGate

from numpy import pi 
def Ansatz(qc, wires, params):


    UU(qc,wires[0:2],params[0:6])
    qc.cx(wires[1],wires[0])
    RR_Z(qc,wires[0:2],params[6:8])
    qc.cx(wires[0],wires[1])
    qc.ry(params[8],wires[1])
    qc.cx(wires[1],wires[0])
    UU(qc,wires[0:2],params[9:15])

    qc.barrier()
    #Module 1
    U_CCR_decom(qc,wires[0:3],params[15:19])

    CUU(qc,[wires[2],wires[0],wires[1]],params[19:25],'0')
    CCX(qc,[wires[1],wires[2], wires[0]],'10')
    CRR_Z(qc,[wires[2],wires[0],wires[1]],params[25:27],'0')
    CCX(qc,[wires[0],wires[2], wires[1]],'10')
    CR_Y(qc,[wires[2], wires[1]],[params[27]],'0')
    CCX(qc,[wires[1],wires[2], wires[0]],'10')
    CUU(qc,[wires[2],wires[0],wires[1]],params[28:34],'0')
    
    CUU(qc,[wires[2],wires[0],wires[1]],params[34:40],'1')
    CCX(qc,[wires[1],wires[2], wires[0]],'11')
    CRR_Z(qc,[wires[2],wires[0],wires[1]],params[40:42],'1')
    CCX(qc,[wires[0],wires[2], wires[1]],'11')
    CR_Y(qc,[wires[2], wires[1]],[params[42]],'1')
    CCX(qc,[wires[1],wires[2], wires[0]],'11')
    CUU(qc,[wires[2],wires[0],wires[1]],params[43:49],'1')

    qc.barrier()
    #Module 2
    U_CCcR_decom(qc, wires[0:4],params[49:53])
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[53:59],'10')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'110')
    CCRR_Z(qc,[wires[2],wires[3],wires[0],wires[1]], params[59:61], '10')
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'110')
    CCR_Y(qc,[wires[2],wires[3],wires[1]],[params[61]],'10')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'110')
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[62:68],'10')
    
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[68:74],'11')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'111')
    CCRR_Z(qc,[wires[2],wires[3],wires[0],wires[1]], params[74:76], '11')
    CCCX(qc,[wires[0],wires[2],wires[3],wires[1]],'111')
    CCR_Y(qc,[wires[2],wires[3],wires[1]],[params[76]],'11')
    CCCX(qc,[wires[1],wires[2],wires[3],wires[0]],'111')
    CCUU(qc,[wires[2],wires[3],wires[0],wires[1]],params[77:83],'11')
    
    CX01(qc, [wires[3],wires[2]])
    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_params=83
params = list(range(num_params))
qc = QuantumCircuit(4)
#PREP(qc)
Ansatz(qc, [0,1,2,3] , params)
qc.draw(output='mpl', style = 'clifford') 

In [ ]:
# ============================================================
# AWS Braket / IQM Garnet setup
# ============================================================
# Required packages on the AWS environment:
#   pip install amazon-braket-sdk qiskit-braket-provider

from braket.aws import AwsDevice
from qiskit_braket_provider import BraketProvider
from qiskit import transpile
from collections import Counter
import numpy as np

GARNET_ARN = "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet"

# Check the device from the AWS account before submitting a paid task.
garnet_device = AwsDevice(GARNET_ARN)
print("Device name   :", garnet_device.name)
print("Device status :", garnet_device.status)
try:
    qd = garnet_device.queue_depth()
    print("Quantum-task queue :", qd.quantum_tasks)
    print("Hybrid-job queue   :", qd.jobs)
except Exception as exc:
    print("Queue depth unavailable:", exc)

provider = BraketProvider()
garnet_backend = provider.get_backend("Garnet")
print("Qiskit-Braket backend:", garnet_backend)

# Safety switch: leave False until the availability/resource checks look correct.
HARDWARE_RUN = True


def _gate_qubit_count(inst):
    try:
        return len(inst.qubits)
    except Exception:
        return 0


def circuit_resource_report(qc, backend, optimization_level=3, seed_transpiler=150):
    """Transpile to Garnet and report circuit/resource complexity.

    Metrics are evaluated on the actual transpiled circuit that will be sent
    through the Qiskit-Braket backend.
    """
    tqc = transpile(
        qc,
        backend=backend,
        optimization_level=optimization_level,
        seed_transpiler=seed_transpiler,
    )

    op_counts = dict(tqc.count_ops())
    one_q = 0
    two_q = 0
    multi_q = 0
    measurements = 0

    for item in tqc.data:
        name = item.operation.name
        nq = len(item.qubits)
        if name == "measure":
            measurements += 1
        elif nq == 1:
            one_q += 1
        elif nq == 2:
            two_q += 1
        elif nq > 2:
            multi_q += 1

    metrics = {
        "num_qubits": tqc.num_qubits,
        "depth": tqc.depth(),
        "size": tqc.size(),
        "1q_gate_count": one_q,
        "2q_gate_count": two_q,
        "multiq_gate_count": multi_q,
        "measurement_count": measurements,
        "cz_count": int(op_counts.get("cz", 0)),
        "cx_count": int(op_counts.get("cx", 0)),
        "operation_counts": op_counts,
    }

    print("\n=== IQM Garnet transpiled resource report ===")
    for key, value in metrics.items():
        print(f"{key:22s}: {value}")

    return tqc, metrics


# Resource check using one representative parameter vector.
# This does NOT submit a QPU task.
resource_params = np.random.default_rng(150).uniform(0, 2*np.pi, 83)
qc_resource = circuit_init()
PREP(qc_resource)
Ansatz(qc_resource, [2, 3, 4, 5], resource_params)
qc_resource = qc_resource.reverse_bits()

garnet_transpiled_example, garnet_resource_metrics = circuit_resource_report(
    qc_resource,
    garnet_backend,
    optimization_level=3,
    seed_transpiler=150,
)


$0.30 per task + $0.00145 per shot.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile
import time


# ============================================================
# Fixed optimal parameters
# ============================================================

optimal_params = np.array([ 0.0282687459,  1.2980352472,  0.3446917772,  1.5312252448,  0.5251518599,
 -0.0563740822,  1.5138493202,  1.0677364023,  0.5928400814,  1.0140451302,
  1.8808885564,  0.2014473021,  0.9564492278,  1.514552884 , -0.0310318803,
  0.0067094371,  3.1974232992,  0.4010087339,  3.0220811641,  2.1491989657,
  2.1930754898,  1.3333274728, -0.0450387894,  0.2019631266,  0.2322093984,
  0.0041757173,  0.5691701078,  0.9258478852,  1.0767614207,  1.1812029198,
  0.2071159783,  0.671525532 ,  0.5828578443,  0.6672049988,  0.8791839438,
  0.4120574452,  1.0268248456,  0.3814718511,  0.4496574088,  1.6857940712,
  0.3019436739,  0.8150841584, -0.075048126 ,  0.4018322958,  0.3040626074,
  1.7395957867,  0.2086044132, -0.2218045946,  1.1509379406,  0.1584383279,
  1.1856618565, -0.0797609024,  3.1479856334,  1.3026079974, -0.33016016  ,
 -0.7632224879,  0.9427524278,  0.2585971527,  0.4311152374,  1.1530602486,
  0.0104697129,  0.9959619965,  1.5515833081,  0.5117751896, -0.0895568064,
  2.0218838591,  0.4537958797,  0.0057961328,  1.5291860393,  1.9349794846,
  0.5181655307,  0.3904673054,  0.1970213004, -0.0964976969,  1.0924749305,
 -0.2647126129,  1.5506429561,  0.5346502201,  0.2014334326,  0.7173739998,
  0.567601884 ,  0.2994904527,  2.7704791979], dtype=float)

print("Number of parameters:", len(optimal_params))

# Should be 83
assert len(optimal_params) == 83

# ============================================================
# AerSimulator fixed-parameter sanity check
# ============================================================

num_shots = 10000
seed_transpiler = 150
seed_simulator = 150

simulator = AerSimulator()


# ============================================================
# Build EXACTLY the same logical circuit
# ============================================================

qc = circuit_init()

qc = PREP(qc)

qc.barrier()

Ansatz(
    qc,
    [2, 3, 4, 5],
    optimal_params
)

qc = qc.reverse_bits()


# ============================================================
# Transpile for AerSimulator
# ============================================================

t_qc = transpile(
    qc,
    backend=simulator,
    optimization_level=3,
    seed_transpiler=seed_transpiler
)

print("\n=== AerSimulator transpiled circuit ===")
print("Depth :", t_qc.depth())
print("Size  :", t_qc.size())
print("Ops   :", t_qc.count_ops())


# ============================================================
# ONE evaluation
# ============================================================

start_time = time.perf_counter()

job = simulator.run(
    t_qc,
    shots=num_shots,
    seed_simulator=seed_simulator
)

result = job.result()

end_time = time.perf_counter()

counts = result.get_counts()

print("\nSimulation wall time:", end_time - start_time, "s")
print("\nCounts:")
print(counts)


# ============================================================
# Calculate P_guess
# EXACTLY SAME outcome mapping as Garnet code
# ============================================================

tar_00 = 0
tar_01 = 0
tar_10 = 0

for outcome, count in counts.items():

    if outcome[0:2] == "00" and outcome[4:6] == "00":
        tar_00 += count

    if outcome[0:2] == "01" and outcome[4:6] == "01":
        tar_01 += count

    if outcome[0:2] == "10" and outcome[4:6] == "10":
        tar_10 += count


p_guess = (
    tar_00
    + tar_01
    + tar_10
) / num_shots


# ============================================================
# Report
# ============================================================

print("\n===================================")
print("Fixed-parameter Aer evaluation")
print("===================================")

print("Shots :", num_shots)

print(
    "tar_00:",
    tar_00,
    "  contribution:",
    tar_00 / num_shots
)

print(
    "tar_01:",
    tar_01,
    "  contribution:",
    tar_01 / num_shots
)

print(
    "tar_10:",
    tar_10,
    "  contribution:",
    tar_10 / num_shots
)

print("-----------------------------------")
print("P_guess =", p_guess)
print("===================================")

In [ ]:
import numpy as np
import time
from qiskit import transpile


# ============================================================
# Settings
# ============================================================

num_shots = 10000
seed_transpiler = 150


# ============================================================
# Build exactly the same circuit
# ============================================================

qc = circuit_init()

qc = PREP(qc)

qc.barrier()

Ansatz(
    qc,
    [2, 3, 4, 5],
    optimal_params
)

qc = qc.reverse_bits()


# ============================================================
# Transpile for IQM Garnet
# ============================================================

t_qc = transpile(
    qc,
    backend=garnet_backend,
    optimization_level=3,
    seed_transpiler=seed_transpiler
)

print("\n=== Transpiled circuit resources ===")
print("Depth :", t_qc.depth())
print("Size  :", t_qc.size())
print("Ops   :", t_qc.count_ops())


# ============================================================
# Run ONE Garnet QPU task
# ============================================================

start_time = time.perf_counter()

job = garnet_backend.run(
    t_qc,
    shots=num_shots,
    verbatim=True
)

print("\nBraket task ID:")
print(job.job_id())

print("\nWaiting for result...")

result = job.result()

end_time = time.perf_counter()

print("Task wall time:", end_time - start_time, "s")


# ============================================================
# Get counts
# ============================================================

counts = result.get_counts()

print("\nCounts:")
print(counts)


# ============================================================
# Calculate guessing probability
# Same definition as your original objective_function
# ============================================================

tar_00 = 0
tar_01 = 0
tar_10 = 0

for outcome, count in counts.items():

    if outcome[0:2] == "00" and outcome[4:6] == "00":
        tar_00 += count

    if outcome[0:2] == "01" and outcome[4:6] == "01":
        tar_01 += count

    if outcome[0:2] == "10" and outcome[4:6] == "10":
        tar_10 += count


p_guess = (
    tar_00
    + tar_01
    + tar_10
) / num_shots


print("\n===================================")
print("Fixed-parameter Garnet evaluation")
print("===================================")

print("Shots :", num_shots)

print(
    "tar_00:",
    tar_00,
    "  contribution:",
    tar_00 / num_shots
)

print(
    "tar_01:",
    tar_01,
    "  contribution:",
    tar_01 / num_shots
)

print(
    "tar_10:",
    tar_10,
    "  contribution:",
    tar_10 / num_shots
)

print("-----------------------------------")
print("P_guess =", p_guess)
print("===================================")